# selectstream-lite: read-side ablations (evaluation only, no training)
Runs the trained `W_A` model on validation QA streams with one part of the reading step changed at a time.
Steps: 0 check the runtime, 1 install, 2 upload code + annotations, 3 clips, 4 Hugging Face files, 5 config, 6 smoke test, 7 ablations, 8 save results.

## Step 0: check the runtime (GPU, disk)

In [1]:
import subprocess
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv; df -h /content | tail -1; free -g | head -2", shell=True, capture_output=True, text=True))

CompletedProcess(args='nvidia-smi --query-gpu=name,memory.total --format=csv; df -h /content | tail -1; free -g | head -2', returncode=0, stdout='name, memory.total [MiB]\nNVIDIA A100-SXM4-40GB, 40960 MiB\noverlay         236G   43G  194G  19% /\n               total        used        free      shared  buff/cache   available\nMem:              83           1          79           0           3          81\n', stderr='')


## Step 1: install pinned packages
`huggingface_hub` must stay below 2.0 (transformers 5.5.4 refuses to import with 2.x).

In [2]:
import subprocess
def sh(c, tail=3000):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-tail:])
sh("pip install -q 'huggingface_hub<2.0' transformers==5.5.4 qwen-vl-utils opencv-python-headless pyyaml 2>&1 | tail -3")
sh("python -c \"import torch,transformers,huggingface_hub as h;print(torch.__version__,transformers.__version__,h.__version__)\"")


2.11.0+cu130 5.5.4 1.33.0



## Step 2: unpack the code and annotations
The repo is private, so everything is uploaded by hand as one file: `colab_upload.zip` into `/content` (Files panel, left sidebar, upload button). It holds the code at commit e6601a9 (`sl/`), the cleaned annotations (`sl/dataset/clean/`) and the Drive ids of the 205 clips (`read_clips_driveids.json`).

In [3]:
import os, subprocess
if not os.path.exists("/content/colab_upload.zip"):
    print("Upload colab_upload.zip to /content first (Files panel, left sidebar)")
else:
    sh("cd /content && rm -rf sl && unzip -q -o colab_upload.zip && ls /content /content/sl /content/sl/dataset/clean")
    sh("cd /content/sl && python -c \"import ast;ast.parse(open('main/cli/eval_ego4d.py').read());print('eval_ego4d.py parses')\"; ls -la /content/read_clips_driveids.json")

/content:
colab_upload.zip
read_clips_driveids.json
sample_data
sl

/content/sl:
configs
dataset
main
README.md
scripts

/content/sl/dataset/clean:
clips.json
nlq_train.json
nlq_val.json
qaego4d_train.json
qaego4d_val.json

eval_ego4d.py parses
-rw-rw-r-- 1 root root 28087 Oct 29  2026 /content/read_clips_driveids.json



## Step 3: download the 205 clips
The clips for the first 150 validation QA streams (about 15 GB), fetched anonymously from the public Drive links. It tries the `ego4d_nlq/clips` copy first and falls back to the original. Clips that are already on disk are skipped.

In [4]:
import json, os, subprocess
from concurrent.futures import ThreadPoolExecutor
ids = json.load(open("/content/read_clips_driveids.json")); os.makedirs("/content/clips", exist_ok=True)
def get(u):
    out = f"/content/clips/{u}.mp4"
    if os.path.exists(out) and os.path.getsize(out) > 1_000_000: return u, "have"
    for k in ("mydrive", "orig"):
        url = f"https://drive.usercontent.google.com/download?id={ids[u][k]}&export=download&confirm=t"
        r = subprocess.run(["curl", "-sSL", "--retry", "3", "-m", "900", "-o", out + ".part", "-w", "%{http_code} %{content_type}", url], capture_output=True, text=True)
        if r.stdout.startswith("200 video") and os.path.getsize(out + ".part") > 1_000_000:
            os.replace(out + ".part", out); return u, k
    return u, "FAILED"
with ThreadPoolExecutor(6) as ex:
    res = list(ex.map(get, ids))
print("clips", len(res), "failed", [u for u, s in res if s == "FAILED"])
sh("du -sh /content/clips; ls /content/clips/*.mp4 | wc -l")

clips 205 failed []
15G	/content/clips
205



## Step 4: Hugging Face files (needs your token)
Run the login cell first and paste your Hugging Face token into the box it shows (the token is not stored in the notebook text). Then the download cell fetches, from the private `lma-project` repos: the `W_A` checkpoint, `segment_center.npy`, the scorer, `calibration.json` (about 1.7 GB), and the feature cache (about 12 GB).

In [5]:
  from huggingface_hub import login
login()   # paste your token in the box that appears

In [6]:
from huggingface_hub import snapshot_download, get_token
tok = get_token(); assert tok, "run the login cell first"
M, F = "lma-project/selectstream-lite-ego4d", "lma-project/selectstream-lite-ego4d-features"
snapshot_download(M, token=tok, local_dir="/content/ck", allow_patterns=[
    "checkpoints/full_v3/W_A/*", "checkpoints/full_v3/R_A/segment_center.npy",
    "runs/full_v3/scorer/scorer.pt", "runs/full_v3/scorer/config.json", "runs/full_v3/calibration.json", "config/*", "README.md"])
snapshot_download(F, repo_type="dataset", token=tok, local_dir="/content/feature_cache")
sh("du -sh /content/ck /content/feature_cache; ls /content/ck/checkpoints/full_v3/W_A /content/feature_cache; ls /content/feature_cache/g | wc -l; ls /content/feature_cache/meta | wc -l")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3374 files:   0%|          | 0/3374 [00:00<?, ?it/s]

1.7G	/content/ck
12G	/content/feature_cache
/content/ck/checkpoints/full_v3/W_A:
weights.pt

/content/feature_cache:
g
meta
README.md
1686
1686



## Step 5: single-GPU config, and a `--start` flag
The cluster config shards the backbone over 4 GPUs; here it is one 40 GB GPU (`max_memory: {0: "38GiB"}`). `eval_ego4d.py` only had `--max_clips N` (first N streams), so a `--start` flag (default 0, so behaviour is unchanged) is added to run later pieces of the stream list.

In [7]:
import re
src = open("/content/sl/configs/ego4d_qwen25vl7b.yaml").read()
new = re.sub(r"max_memory:\s*\{[^}]*\}", 'max_memory: {0: "38GiB"}', src); assert new != src
open("/content/sl/configs/ego4d_colab.yaml", "w").write(new)
p = "/content/sl/main/cli/eval_ego4d.py"; s = open(p).read()
if "--start" not in s:
    s = s.replace('ap.add_argument("--max_clips", type=int, default=None, help="First N streams.")',
                  'ap.add_argument("--max_clips", type=int, default=None, help="N streams (from --start).")\n    ap.add_argument("--start", type=int, default=0, help="Skip the first START streams (run a later piece).")')
    s = s.replace("ds.items = ds.items[: args.max_clips]", "ds.items = ds.items[args.start: args.start + args.max_clips]")
    open(p, "w").write(s)
sh("grep -n 'args.start' " + p + "; grep -n max_memory /content/sl/configs/ego4d_colab.yaml")

120:        ds.items = ds.items[args.start: args.start + args.max_clips]
8:  max_memory: {0: "38GiB"}



## Step 6: runner and smoke test
`run_piece(start, n, name, flags)` evaluates arm A (`W_A`) on validation QA streams `start..start+n-1` with one reading-step change (`flags`), and writes `summary.json` and `predictions.jsonl` to `/content/out/`. No training happens. The smoke test runs the unchanged reference on 3 streams to check paths, model loading and output.

In [8]:
import os, subprocess, time, json
def run_piece(start, n, name, flags=(), root="/content/out"):
    tag = f"s{start:03d}_{start + n:03d}"; out = f"{root}/{tag}/read_{name}"
    if os.path.exists(out + "/summary.json"): return "skip"
    os.makedirs(f"{root}/{tag}", exist_ok=True)
    cmd = ["python", "-m", "main.cli.eval_ego4d", "--config", "configs/ego4d_colab.yaml", "--arm", "A", "--ckpt", "/content/ck/checkpoints/full_v3/W_A",
           "--track", "qa", "--video_dir", "/content/clips", "--feature_cache", "/content/feature_cache", "--scorer", "/content/ck/runs/full_v3/scorer/scorer.pt",
           "--calibration", "/content/ck/runs/full_v3/calibration.json", "--clean_dir", "/content/sl/dataset/clean", "--out_dir", out,
           "--start", str(start), "--max_clips", str(n), *flags]
    with open(f"{root}/{tag}/log_{name}.txt", "w") as f:
        rc = subprocess.run(cmd, cwd="/content/sl", env={**os.environ, "PYTHONPATH": "/content/sl", "PYTHONUNBUFFERED": "1"}, stdout=f, stderr=subprocess.STDOUT).returncode
    return "ok" if os.path.exists(out + "/summary.json") else f"FAILED rc={rc}"
t = time.time(); print(run_piece(0, 3, "full", root="/content/out_smoke"), f"{time.time() - t:.0f}s")
sh("tail -c 700 /content/out_smoke/s000_003/log_full.txt; cat /content/out_smoke/s000_003/read_full/summary.json")

ok 210s
,
   "device_map": "auto",
   "trust_remote_code": true,
   "max_memory": {
    "0": "38GiB"
   }
  },
  "vismem": {
   "short_invoke_token": "<ms_I>",
   "short_end_token": "<ms_E>",
   "long_invoke_token": "<ml_I>",
   "long_end_token": "<ml_E>",
   "query_len": 8,
   "short_mem_len": 8,
   "long_mem_len": 16,
   "query_builder": {
    "num_layers": 1,
    "num_heads": 8,
    "dropout": 0.0,
    "ff_mult": 4
   },
   "former_backend": "lvm_sft",
   "visual_feature_source": "projected",
   "trainable_dtype": "float32",
   "trainable_device": "cuda:0",
   "lvm_num_slots": 16,
   "method_graph_budget": 8,
   "method_seed_topk": 4,
   "method_evidence_tokens": 4,
   "lvm_segment_len": 32,
   "lvm_tau_r": 0.9896904706954956,
   "lvm_tau_s": 0.06815026234835386,
   "lvm_gate_hidden": 256,
   "lvm_normalize_slot": true,
   "lvm_eviction_policy": "fifo",
   "lvm_readout_policy": "recent",
   "lvm_delta_t_scale": 32.0,
   "lvm_enable_graph_merge": true,
   "lvm_merge_similarity_weight

## Step 7: the ablations, in short pieces
18 variants (the reference, 3 no-memory / no-frame checks, 2 retrieval bounds, 5 ranking-score terms, 3 graph parts, evidence counts 1/2/8, and the open segment), each on pieces of 10 validation streams. The loop goes piece by piece (all 18 variants on streams 0-9, then 10-19, ...), so any stop leaves a complete grid on the finished pieces. It stops cleanly after `BUDGET_MIN` minutes; run the cell again to continue (finished variants are skipped).

In [9]:
VARIANTS = [
    ("full", []), ("text_only", ["--blank_frame", "--evidence_k", "0"]), ("blank_frame", ["--blank_frame"]), ("M0", ["--evidence_k", "0"]),
    ("bound_random", ["--read_ablation", "random"]), ("bound_oracle", ["--read_ablation", "oracle"]),
    ("score_no_prior", ["--read_ablation", "no_prior"]), ("score_no_cos", ["--read_ablation", "no_cos"]), ("score_no_bias", ["--read_ablation", "no_bias"]),
    ("score_cos_only", ["--read_ablation", "cos_only"]), ("score_prior_only", ["--read_ablation", "prior_only"]),
    ("graph_no_hops", ["--read_ablation", "no_hops"]), ("graph_no_gar", ["--read_ablation", "no_gar"]), ("graph_coarse_only", ["--read_ablation", "coarse_only"]),
    ("M1", ["--evidence_k", "1"]), ("M2", ["--evidence_k", "2"]), ("M8", ["--evidence_k", "8"]), ("no_open", ["--no_open"]),
]
TOTAL, CHUNK, BUDGET_MIN = 50, 10, 45
t0 = time.time()
for a in range(0, TOTAL, CHUNK):
    for name, flags in VARIANTS:
        if time.time() - t0 > BUDGET_MIN * 60: break
        t = time.time(); r = run_piece(a, CHUNK, name, flags)
        if r != "skip": print(f"{time.strftime('%H:%M:%S')} s{a:03d}_{a + CHUNK:03d} {name}: {r} ({(time.time() - t) / 60:.1f} min)", flush=True)
    else: continue
    break
done = sum(os.path.exists(f"/content/out/s{a:03d}_{a + CHUNK:03d}/read_{n}/summary.json") for a in range(0, TOTAL, CHUNK) for n, _ in VARIANTS)
print(f"finished {done}/{len(VARIANTS) * (TOTAL // CHUNK)} pieces")

15:38:34 s000_010 full: ok (7.0 min)
15:45:56 s000_010 text_only: ok (7.4 min)
15:52:54 s000_010 blank_frame: ok (7.0 min)
16:00:24 s000_010 M0: ok (7.5 min)
16:07:21 s000_010 bound_random: ok (6.9 min)
16:14:18 s000_010 bound_oracle: ok (6.9 min)
16:21:14 s000_010 score_no_prior: ok (6.9 min)
finished 7/90 pieces


## Step 7b: the same loop, in a background thread
The Step 7 cell blocks the notebook, and the MCP connection gives up after 30 silent minutes. This version starts the loop in a background thread and returns at once; the next cell reports progress while it runs. Finished variants are skipped, so it picks up where Step 7 stopped. Run `STATE["stop"] = True` to stop it after the current piece.

In [10]:
import threading
STATE = {"log": [], "stop": False, "cur": None, "t0": time.time()}
def worker():
    for a in range(0, TOTAL, CHUNK):
        for name, flags in VARIANTS:
            if STATE["stop"]: STATE["cur"] = "stopped"; return
            STATE["cur"] = f"s{a:03d}_{a + CHUNK:03d} {name}"; t = time.time()
            r = run_piece(a, CHUNK, name, flags)
            if r != "skip": STATE["log"].append(f"{time.strftime('%H:%M:%S')} UTC {STATE['cur']}: {r} ({(time.time() - t) / 60:.1f} min)")
    STATE["cur"] = "ALL DONE"
th = threading.Thread(target=worker, daemon=True); th.start()
print("started background loop")

started background loop


### Progress (run any time)

In [98]:
import glob
n = len(glob.glob("/content/out/s*/read_*/summary.json")); tot = len(VARIANTS) * (TOTAL // CHUNK)
print(f"{time.strftime('%H:%M:%S')} UTC  finished {n}/{tot} pieces  current: {STATE['cur']}  thread alive: {th.is_alive()}")
print("\n".join(STATE["log"][-8:]))
sh("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; df -h /content | tail -1")

03:29:02 UTC  finished 90/90 pieces  current: ALL DONE  thread alive: False
02:40:29 UTC s040_050 score_prior_only: ok (5.3 min)
02:45:50 UTC s040_050 graph_no_hops: ok (5.3 min)
02:51:13 UTC s040_050 graph_no_gar: ok (5.4 min)
02:56:36 UTC s040_050 graph_coarse_only: ok (5.4 min)
03:01:59 UTC s040_050 M1: ok (5.4 min)
03:07:23 UTC s040_050 M2: ok (5.4 min)
03:13:03 UTC s040_050 M8: ok (5.7 min)
03:18:24 UTC s040_050 no_open: ok (5.3 min)
0 %, 0 MiB
overlay         236G   87G  150G  37% /



### Results so far (compact)
Prints one JSON line per finished piece (metrics from each `summary.json`), so the numbers can be copied out even if the VM is lost.

In [85]:
import glob, json, os
rows = []
KEYS = ("n", "recall_at_m", "t_overlap", "evidence_precision", "best_iou", "exact", "contains", "evidence_survival", "evidence_span_s", "num_committed")
for f in sorted(glob.glob("/content/out/s*/read_*/summary.json")):
    d = json.load(open(f)); piece = f.split("/")[3]; name = f.split("/")[4][5:]
    o = d.get("groups", {}).get("overall", {})
    rows.append({"piece": piece, "variant": name, **{k: (round(o[k], 4) if isinstance(o[k], float) else o[k]) for k in KEYS if k in o}})
print(len(rows), "pieces")
for r in rows: print(json.dumps(r))

55 pieces
{"piece": "s000_010", "variant": "M0", "n": 130, "recall_at_m": 0.0, "t_overlap": 0.0, "evidence_precision": null, "best_iou": 0.2345, "exact": 0.0, "contains": 0.1769, "evidence_survival": 1.0, "evidence_span_s": null, "num_committed": 14.5462}
{"piece": "s000_010", "variant": "M1", "n": 130, "recall_at_m": 0.1538, "t_overlap": 0.1262, "evidence_precision": 0.0446, "best_iou": 0.2353, "exact": 0.1692, "contains": 0.2385, "evidence_survival": 1.0, "evidence_span_s": 39.5602, "num_committed": 14.5462}
{"piece": "s000_010", "variant": "M2", "n": 130, "recall_at_m": 0.2692, "t_overlap": 0.2125, "evidence_precision": 0.0411, "best_iou": 0.2303, "exact": 0.1538, "contains": 0.2385, "evidence_survival": 1.0, "evidence_span_s": 36.7491, "num_committed": 14.5462}
{"piece": "s000_010", "variant": "M8", "n": 130, "recall_at_m": 0.8, "t_overlap": 0.7392, "evidence_precision": 0.0294, "best_iou": 0.2204, "exact": 0.1462, "contains": 0.2385, "evidence_survival": 1.0, "evidence_span_s": 79

## Step 9: keep a copy of the results on Google Drive
Mounts Drive, creates `MyDrive/selectstream_ablations/`, and copies `/content/out` into it. The Colab disk is wiped if the runtime is dropped, so a copy outside the VM matters. Run the copy cell again (or let the sync thread do it) to refresh.

In [16]:
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/selectstream_ablations"
os.makedirs(DEST, exist_ok=True)
def sync():
    r = subprocess.run(f"rsync -a /content/out/ {DEST}/out/ && cp -f /content/sl/configs/ego4d_colab.yaml {DEST}/", shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-300:]
print(sync())
sh(f"ls {DEST}; du -sh {DEST}")

Mounted at /content/drive
(0, '')
ego4d_colab.yaml
out
1.1M	/content/drive/MyDrive/selectstream_ablations



In [17]:
print("mounted:", os.path.ismount("/content/drive"))
sh("ls /content/drive 2>&1 | head; ls /content/drive/MyDrive 2>&1 | head -20; ls /content/drive/MyDrive/selectstream_ablations 2>&1 | head")
print(f"{time.strftime('%H:%M:%S')} UTC finished {len(glob.glob('/content/out/s*/read_*/summary.json'))}/90, current {STATE['cur']}, thread alive {th.is_alive()}")

mounted: True
MyDrive
ego4d_colab.yaml
out
16:29:28 UTC finished 8/90, current s000_010 score_no_bias, thread alive True
[Drive folder listing removed: it showed unrelated personal files]


### Background sync to Drive (every 10 min)
Copies `/content/out` into `MyDrive/selectstream_ablations/out/` only; nothing else on the Drive is touched. Does a final sync when the loop reports `ALL DONE`.

In [18]:
SYNC = {"last": None, "stop": False}
def sync_loop():
    while not SYNC["stop"]:
        time.sleep(600)
        SYNC["last"] = (time.strftime("%H:%M:%S"), sync())
        if STATE["cur"] in ("ALL DONE", "stopped"): SYNC["last"] = (time.strftime("%H:%M:%S"), sync(), "final"); return
sth = threading.Thread(target=sync_loop, daemon=True); sth.start()
print("sync thread started; first copy happens within 10 min. Last sync was done by the mount cell.")
print(sync())
sh(f"du -sh {DEST}/out; find {DEST}/out -name summary.json | wc -l")

sync thread started; first copy happens within 10 min. Last sync was done by the mount cell.
(0, '')
1.2M	/content/drive/MyDrive/selectstream_ablations/out
8



In [46]:
vm = sorted(glob.glob("/content/out/s*/read_*/summary.json"))
dr = sorted(glob.glob(DEST + "/out/s*/read_*/summary.json"))
print(f"{time.strftime('%H:%M:%S')} UTC  on VM disk: {len(vm)} pieces | copied to Drive: {len(dr)} pieces | sync thread alive: {sth.is_alive()} | last sync: {SYNC['last']}")
print("missing from Drive:", [p.split('/out/')[1] for p in vm if DEST + "/out/" + p.split('/out/')[1] not in dr][:5])
sh(f"du -sh /content/out {DEST}/out; ls -la --time-style=+%H:%M {DEST}/out/s010_020/read_bound_oracle/ | tail -3")

19:04:17 UTC  on VM disk: 27 pieces | copied to Drive: 27 pieces | sync thread alive: True | last sync: ('18:59:48', (0, ''))
missing from Drive: []
4.4M	/content/out
4.1M	/content/drive/MyDrive/selectstream_ablations/out
drwx------ 12 root root   4096 18:58 ..
-rw-------  1 root root 168858 18:32 predictions.jsonl
-rw-------  1 root root  12679 18:32 summary.json



In [54]:
import sys, yaml
sys.path.insert(0, "/content/sl")
from main.data.ego4d_qa import Ego4DStreamDataset
c = yaml.safe_load(open("/content/sl/configs/ego4d_colab.yaml")); d = c["data"]; t = c["training"]
ds = Ego4DStreamDataset(clean_dir="/content/sl/dataset/clean", video_dir="/content/clips", split="val", source="qaego4d", part="all",
    dev_frac=float(d.get("dev_frac", 0.1)), query_time_mode=str(d.get("query_time_mode", "uniform")), sample_fps=float(d.get("sample_fps", 1.0)),
    resample_ask_times=False, seed=int(t.get("seed", 42)), require_video=False)
for a in range(0, 50, 10):
    it = ds.items[a:a + 10]; print(f"streams {a}-{a + 9}: frames {sum(i.num_frames for i in it)}, queries {sum(len(i.queries) for i in it)}")

streams 0-9: frames 5852, queries 130
streams 10-19: frames 8473, queries 173
streams 20-29: frames 9134, queries 295
streams 30-39: frames 6608, queries 159
streams 40-49: frames 4770, queries 110


In [55]:
import collections
rows = [json.loads(l) for l in open("/content/out/s000_010/read_full/predictions.jsonl")]
print(len(rows), "rows; keys:", list(rows[0].keys()))
r = rows[0]; print({k: (v if not isinstance(v, (list, dict)) or len(str(v)) < 120 else str(v)[:120]) for k, v in r.items()})
for k in ("template", "delay_bucket", "evidence_location", "linked", "participant_disjoint"):
    if k in r: print(k, collections.Counter(x.get(k) for x in rows).most_common(8))
allr = [json.loads(l) for f in glob.glob("/content/out/s*/read_full/predictions.jsonl") for l in open(f)]
print("full rows over all finished pieces:", len(allr), "| templates:", sorted({x.get('template') for x in allr}), "| streams:", len({x.get('stream_id') for x in allr}))

130 rows; keys: ['query_id', 'stream_id', 'clip_uid', 'track', 'arm', 'template', 'linked', 'nlq_query_id', 'participant_disjoint', 'window', 'ask_time', 'delay_bucket', 'evidence_location', 'question', 'answer', 'prediction', 'evidence_spans', 'best_iou', 'recall_at_m', 't_overlap', 'evidence_survival', 'num_committed', 'num_evidence', 'evidence_span_s', 'evidence_precision', 'exact', 'contains']
{'query_id': 'qa:a60f6dff-a926-4182-b5ca-adf421c077d4_1', 'stream_id': '000cd456-ff8d-499b-b0c1-4acead128a8b', 'clip_uid': 'bfea3e01-55f4-4744-a7c6-4494c7b99ef2', 'track': 'qa', 'arm': 'A', 'template': 'Objects: Where is object X before / after event Y?', 'linked': True, 'nlq_query_id': 'nlq:88434fde-db65-41a5-bbaa-39a279264e23_3', 'participant_disjoint': False, 'window': [520.8977030791666, 522.7927030791666], 'ask_time': 539.0207030791667, 'delay_bucket': '[16,64)s', 'evidence_location': 'open', 'question': 'where was the trolley before i dragged it?', 'answer': 'at the floor', 'prediction'

TypeError: '<' not supported between instances of 'str' and 'NoneType'

## Step 10: plots (run after the ablations finish; saved only to `MyDrive/selectstream_ablations/plots/`)
Questions are paired across variants (same query ids). Error bars are 95% bootstrap intervals that resample whole streams, so questions from one video are not treated as independent. Colours follow the entity: full = blue, text only = orange, oracle evidence = aqua, random evidence = gray. Only pieces where all 18 variants finished are used.

In [101]:
import pandas as pd, numpy as np, matplotlib, re, string
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
PLOTS = DEST + "/plots"; os.makedirs(PLOTS, exist_ok=True)
VAR_ORDER = [n for n, _ in VARIANTS]
NICE = {"full": "full model (reference)", "text_only": "text only (no frame, no memory)", "blank_frame": "blank frame", "M0": "no memory (0 tokens)",
        "M1": "1 evidence token", "M2": "2 evidence tokens", "M8": "8 evidence tokens", "bound_random": "random evidence", "bound_oracle": "oracle evidence",
        "score_no_prior": "ranking: no prior", "score_no_cos": "ranking: no cosine", "score_no_bias": "ranking: no bias terms", "score_cos_only": "ranking: cosine only",
        "score_prior_only": "ranking: prior only", "graph_no_hops": "graph: no hops", "graph_no_gar": "graph: no reasoning layers",
        "graph_coarse_only": "graph: first pass only", "no_open": "open segment hidden"}
# colour tokens (validated palette; entity -> colour)
SURF, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, RED, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#898781"
SEQ = LinearSegmentedColormap.from_list("seqblue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#0d366b"])
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF, "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": INK2, "axes.edgecolor": AXIS, "axes.linewidth": 1, "grid.color": GRID, "grid.linewidth": 1, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "semibold", "axes.titlecolor": INK,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 100})

def load_all():
    recs = []
    for f in glob.glob("/content/out/s*/read_*/predictions.jsonl"):
        piece, var = f.split("/")[3], f.split("/")[4][5:]
        for l in open(f):
            r = json.loads(l); r["variant"], r["piece"] = var, piece; recs.append(r)
    df = pd.DataFrame(recs)
    cnt = df.groupby("piece")["variant"].nunique(); ok = cnt[cnt == len(VAR_ORDER)].index
    df = df[df.piece.isin(ok)].copy()
    for c in ["recall_at_m", "t_overlap", "evidence_precision", "best_iou", "exact", "contains", "evidence_span_s"]: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df, sorted(ok)
DF, PIECES = load_all()
QS = DF.drop_duplicates("query_id").set_index("query_id")["stream_id"]       # query -> stream
QM = DF.drop_duplicates("query_id").set_index("query_id")                    # per-query attributes
def piv(metric): return DF.pivot_table(index="query_id", columns="variant", values=metric, aggfunc="first")
P = {m: piv(m) for m in ["recall_at_m", "contains", "exact", "best_iou", "t_overlap"]}
PRED = DF.pivot_table(index="query_id", columns="variant", values="prediction", aggfunc="first")
rng = np.random.default_rng(0)
def boot(x, B=2000):
    """mean and 95% CI of a per-question series x (index = query_id), resampling whole streams"""
    x = x.dropna(); g = x.groupby(QS.reindex(x.index).values)
    S, N = g.sum().values, g.count().values
    idx = rng.integers(0, len(S), (B, len(S)))
    m = S[idx].sum(1) / N[idx].sum(1)
    return S.sum() / N.sum(), *np.percentile(m, [2.5, 97.5])
def delta(metric, v, ref="full"): return boot(P[metric][v] - P[metric][ref])
def check_layout(fig):
    fig.canvas.draw(); r = fig.canvas.get_renderer(); W, H = fig.canvas.get_width_height(); items = []
    for ax in fig.axes:
        items += ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels()
        if ax.get_legend(): items += ax.get_legend().get_texts()
    items += fig.texts
    boxes = [(t.get_text(), t.get_window_extent(r)) for t in items if t.get_text().strip() and t.get_visible()]
    bad = []
    for i, (a, ba) in enumerate(boxes):
        if ba.x0 < -2 or ba.y0 < -2 or ba.x1 > W + 2 or ba.y1 > H + 2: bad.append(f"clipped: {a[:30]!r}")
        for b, bb in boxes[i + 1:]:
            if ba.overlaps(bb) and ba.width > 0 and bb.width > 0:
                ox = min(ba.x1, bb.x1) - max(ba.x0, bb.x0); oy = min(ba.y1, bb.y1) - max(ba.y0, bb.y0)
                if ox > 2 and oy > 2: bad.append(f"overlap: {a[:22]!r} / {b[:22]!r}")
    return bad
SAVED = {}
def save(fig, name, caption):
    bad = check_layout(fig); fig.savefig(f"{PLOTS}/{name}.png", dpi=200, bbox_inches="tight"); plt.close(fig); SAVED[name] = caption
    print(f"saved {name}.png" + (f"  LAYOUT: {bad[:6]}" if bad else "  (layout ok)"))
print(len(DF), "rows,", DF.variant.nunique(), "variants, pieces used:", PIECES, "| questions:", DF.query_id.nunique(), "| streams:", QS.nunique())

15606 rows, 18 variants, pieces used: ['s000_010', 's010_020', 's020_030', 's030_040', 's040_050'] | questions: 867 | streams: 50


In [102]:
import textwrap
import matplotlib.text as mtext
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
def check_layout(fig):
    """text boxes that overlap each other. Hidden tick labels outside the axis range are ignored; an annotation counts only by its own text, not by its leader line."""
    fig.canvas.draw(); r = fig.canvas.get_renderer(); items = []
    for ax in fig.axes:
        x0, x1 = sorted(ax.get_xlim()); y0, y1 = sorted(ax.get_ylim())
        items += ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]
        items += [l for l, loc in zip(ax.get_xticklabels(), ax.get_xticks()) if x0 - 1e-9 <= loc <= x1 + 1e-9]
        items += [l for l, loc in zip(ax.get_yticklabels(), ax.get_yticks()) if y0 - 1e-9 <= loc <= y1 + 1e-9]
        if ax.get_legend(): items += ax.get_legend().get_texts()
    items += fig.texts + [t for lg in fig.legends for t in lg.get_texts()]
    boxes = [(t.get_text(), mtext.Text.get_window_extent(t, r)) for t in items if t.get_text().strip() and t.get_visible()]
    bad = []
    for i, (a, ba) in enumerate(boxes):
        for b, bb in boxes[i + 1:]:
            ox = min(ba.x1, bb.x1) - max(ba.x0, bb.x0); oy = min(ba.y1, bb.y1) - max(ba.y0, bb.y0)
            if ox > 1 and oy > 1: bad.append(f"overlap: {a[:24]!r} / {b[:24]!r}")
    return bad
def tidy_x(axes, n=5):
    for ax in np.atleast_1d(axes): ax.xaxis.set_major_locator(MaxNLocator(nbins=n, prune="both"))
GROUPS = [("removed", ["text_only", "blank_frame", "M0"]), ("evidence count", ["M1", "M2", "M8"]), ("bounds", ["bound_random", "bound_oracle"]),
          ("ranking", ["score_no_prior", "score_no_cos", "score_no_bias", "score_cos_only", "score_prior_only"]),
          ("graph", ["graph_no_hops", "graph_no_gar", "graph_coarse_only"]), ("open", ["no_open"])]
ROWS = [v for _, vs in GROUPS for v in vs]
def seps(ax, n):
    b = 0
    for _, vs in GROUPS[:-1]:
        b += len(vs); ax.axhline(n - b - 0.5, color=GRID, lw=1, zorder=0)
def dot(ax, x, lo, hi, y, col):
    ax.plot([lo, hi], [y, y], color=col, lw=2, solid_capstyle="round", zorder=2)
    ax.plot(x, y, "o", ms=8, color=col, mec=SURF, mew=2, zorder=3)
def hdr(fig, title, sub=None):
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="semibold", y=1.0)
    if sub: fig.text(0.01, 0.955, sub, ha="left", fontsize=9.5, color=INK2)
def legend_dots(fig, labels, cols, y=-0.03):
    fig.legend([Line2D([], [], marker="o", ms=8, color=c, lw=2, mec=SURF, mew=2) for c in cols], labels, loc="lower center", ncol=len(cols), bbox_to_anchor=(0.5, y))
NQ, NS = DF.query_id.nunique(), QS.nunique(); errs = []
# ---- Figure 1: change vs the full model, three metrics side by side
try:
    fig, axes = plt.subplots(1, 3, figsize=(14, 7.4), sharey=True); n = len(ROWS); big = {}
    for ax, (m, title) in zip(axes, [("recall_at_m", "Retrieval: Recall@M"), ("contains", "Answer: contains-match"), ("exact", "Answer: exact-match")]):
        res = {v: delta(m, v) for v in ROWS}; big[m] = max(res.values(), key=lambda t: abs(t[0]))[0]
        for i, v in enumerate(ROWS):
            mu, lo, hi = res[v]; dot(ax, mu, lo, hi, n - 1 - i, BLUE if (lo > 0 or hi < 0) else GRAY)
        for v in sorted(ROWS, key=lambda v: -abs(res[v][0]))[:2]:
            ax.annotate(f"{res[v][0]:+.2f}", (res[v][0], n - 1 - ROWS.index(v)), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=9, color=INK)
        ax.axvline(0, color=AXIS, lw=1, zorder=1); seps(ax, n); ax.grid(axis="x"); ax.set_axisbelow(True); ax.set_title(title, loc="left")
        ax.set_yticks(range(n)); ax.set_yticklabels([NICE[v] for v in ROWS][::-1]); ax.set_ylim(-0.7, n - 0.3); ax.set_xlabel("change vs full model")
        ax.xaxis.set_major_formatter(lambda x, p: "0" if abs(x) < 1e-9 else f"{x:+.2f}")
    tidy_x(axes, 4)
    hdr(fig, "What changes when one part of the reader is altered (variant minus full model)",
        f"Largest change: Recall@M {big['recall_at_m']:+.2f}, contains {big['contains']:+.2f}, exact {big['exact']:+.2f}.  {NQ} questions, {NS} validation streams; bars are 95% bootstrap intervals over streams.")
    legend_dots(fig, ["95% interval excludes 0", "95% interval includes 0 (no clear change)"], [BLUE, GRAY])
    fig.tight_layout(rect=(0, 0.02, 1, 0.94)); fig.subplots_adjust(wspace=0.12)
    save(fig, "01_change_vs_full", "Variant minus full model for Recall@M, contains-match and exact-match, with 95% bootstrap intervals over streams.")
except Exception as e: errs.append(("fig1", repr(e)))
# ---- Figure 2: retrieval vs answer accuracy, one point per variant
try:
    grp = {v: "reading-step ablations (incl. full)" for v in ["full"] + GROUPS[3][1] + GROUPS[4][1] + GROUPS[5][1]}
    grp.update({v: "memory or frame removed" for v in GROUPS[0][1]}); grp.update({v: "evidence count and bounds" for v in GROUPS[1][1] + GROUPS[2][1]})
    col = {"reading-step ablations (incl. full)": BLUE, "memory or frame removed": ORANGE, "evidence count and bounds": AQUA}
    st = {v: (boot(P["recall_at_m"][v]), boot(P["contains"][v])) for v in VAR_ORDER}
    fig, ax = plt.subplots(figsize=(9.6, 6.6))
    for v in VAR_ORDER:
        (rx, rl, rh), (cy, cl, ch) = st[v]; c = col[grp[v]]
        ax.plot([rl, rh], [cy, cy], color=c, lw=1, alpha=0.5, zorder=1); ax.plot([rx, rx], [cl, ch], color=c, lw=1, alpha=0.5, zorder=1)
        ax.plot(rx, cy, "o", ms=8, color=c, mec=SURF, mew=2, zorder=3)
    xs = np.array([st[v][0][0] for v in VAR_ORDER]); ys = np.array([st[v][1][0] for v in VAR_ORDER]); sl, ic = np.polyfit(xs, ys, 1)
    ax.plot([0, 1], [ic, sl + ic], color=GRAY, lw=1.5, zorder=0)
    ytop = max(st[v][1][2] for v in VAR_ORDER) * 1.25; ax.set_ylim(0, ytop); ax.set_xlim(-0.03, 1.05)
    off = {"text_only": (9, -14, "left", "text only"), "M0": (9, 7, "left", "no memory"), "M1": (0, 12, "center", "1 token"), "M2": (0, 12, "center", "2 tokens"),
           "M8": (0, -20, "center", "8 tokens"), "bound_oracle": (-10, 12, "right", "oracle evidence"), "bound_random": (-10, -18, "right", "random evidence"), "blank_frame": (10, -16, "left", "blank frame")}
    for v, (dx, dy, ha, txt) in off.items():
        ax.annotate(txt, (st[v][0][0], st[v][1][0]), xytext=(dx, dy), textcoords="offset points", ha=ha, fontsize=9, color=INK)
    fx, fy = st["full"][0][0], st["full"][1][0]       # the cluster of full model + ranking/graph variants: label straight above it, short vertical leader
    ax.annotate("full model and 9 ranking/graph variants", (fx, fy), xytext=(fx - 0.05, ytop * 0.93), textcoords="data", ha="left", fontsize=9, color=INK,
                arrowprops=dict(arrowstyle="-", color=INK2, lw=0.8, shrinkA=2, shrinkB=8))
    ax.grid(True); ax.set_axisbelow(True); ax.set_xlabel("Recall@M: how often the answer moment is among the retrieved evidence"); ax.set_ylabel("contains-match accuracy")
    ax.set_title("Retrieval quality against answer accuracy, one point per variant", loc="left", fontsize=12, pad=40)
    ax.text(0.0, 1.015, textwrap.fill(f"Across variants Recall@M spans {xs.min():.2f}-{xs.max():.2f} and contains-match {ys.min():.2f}-{ys.max():.2f}; trend line {sl:+.2f} contains per unit of Recall@M ({NQ} questions, 95% intervals over streams).", 100), transform=ax.transAxes, fontsize=8.5, color=INK2)
    ax.legend([Line2D([], [], marker="o", ms=8, color=c, lw=0, mec=SURF, mew=2) for c in col.values()], list(col.keys()), loc="lower right")
    fig.tight_layout(); save(fig, "02_retrieval_vs_answer", "Each variant: Recall@M against contains-match accuracy, with a linear trend line.")
except Exception as e: errs.append(("fig2", repr(e)))
# ---- Figure 3: evidence-budget curve
try:
    MS = {0: "M0", 1: "M1", 2: "M2", 4: "full", 8: "M8"}; fig, ax = plt.subplots(figsize=(8.6, 5.4))
    for m, name, c in [("recall_at_m", "Recall@M", BLUE), ("contains", "contains-match", ORANGE), ("exact", "exact-match", AQUA)]:
        r = [boot(P[m][v]) for v in MS.values()]; x = list(MS.keys()); mu = [t[0] for t in r]
        ax.fill_between(x, [t[1] for t in r], [t[2] for t in r], color=c, alpha=0.10, lw=0)
        ax.plot(x, mu, "-o", color=c, lw=2, ms=8, mec=SURF, mew=2, label=name, zorder=3)
        ax.annotate(f"{mu[-1]:.2f}", (8, mu[-1]), xytext=(9, 0), textcoords="offset points", va="center", fontsize=9, color=INK)
    ax.axvline(4, color=AXIS, lw=1, zorder=0); ax.text(4.1, 0.98, "trained with 4", fontsize=8.5, color=MUTED, va="top")
    ax.set_xticks(list(MS.keys())); ax.set_xlim(-0.3, 8.9); ax.set_ylim(0, 1); ax.grid(axis="y"); ax.set_axisbelow(True)
    ax.set_xlabel("evidence tokens passed to the answerer"); ax.set_ylabel("proportion of questions"); ax.legend(loc="upper left")
    ax.set_title("Recall@M, contains-match and exact-match as the evidence budget grows", loc="left", fontsize=12)
    fig.tight_layout(); save(fig, "03_evidence_budget", "Recall@M, contains-match and exact-match against the number of evidence tokens (0, 1, 2, 4 = full, 8).")
except Exception as e: errs.append(("fig3", repr(e)))
# ---- Figure 4: random vs full vs oracle evidence
try:
    vs = ["bound_random", "full", "bound_oracle"]; fig, ax = plt.subplots(figsize=(7.6, 5.2)); w = 0.12
    for k, (m, name, c) in enumerate([("recall_at_m", "Recall@M", BLUE), ("contains", "contains-match", ORANGE)]):
        for i, v in enumerate(vs):
            mu, lo, hi = boot(P[m][v]); x = i + (k - 0.5) * (w + 0.03)
            ax.bar(x, mu, width=w, color=c, zorder=2, label=name if i == 0 else None); ax.plot([x, x], [lo, hi], color=INK2, lw=1, zorder=3)
            ax.annotate(f"{mu:.2f}", (x, hi), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9, color=INK)
    ax.set_xticks(range(3)); ax.set_xticklabels(["random evidence", "full model", "oracle evidence"]); ax.set_ylim(0, 1.1); ax.grid(axis="y"); ax.set_axisbelow(True)
    ax.set_ylabel("proportion of questions"); ax.legend(loc="upper left"); ax.set_title("Random, learned and oracle evidence: retrieval and answer accuracy", loc="left", fontsize=12)
    fig.tight_layout(); save(fig, "04_random_full_oracle", "Recall@M and contains-match with random, learned (full) and oracle evidence.")
except Exception as e: errs.append(("fig4", repr(e)))
# ---- Figure 5: component ablations in absolute terms
try:
    rows5 = ["full"] + GROUPS[3][1] + GROUPS[4][1] + GROUPS[5][1]; n = len(rows5)
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 5.6), sharey=True)
    for ax, (m, title) in zip(axes, [("recall_at_m", "Retrieval: Recall@M"), ("best_iou", "Retrieval: best-node IoU"), ("contains", "Answer: contains-match")]):
        ref = boot(P[m]["full"])[0]; ax.axvline(ref, color=BLUE, lw=1, alpha=0.4, zorder=0)
        for i, v in enumerate(rows5):
            mu, lo, hi = boot(P[m][v]); dot(ax, mu, lo, hi, n - 1 - i, BLUE if v == "full" else ORANGE)
        for y in (n - 1.5, n - 6.5, n - 9.5): ax.axhline(y, color=GRID, lw=1, zorder=0)
        ax.grid(axis="x"); ax.set_axisbelow(True); ax.set_title(title, loc="left"); ax.set_yticks(range(n)); ax.set_yticklabels([NICE[v] for v in rows5][::-1]); ax.set_ylim(-0.7, n - 0.3)
    tidy_x(axes, 4); hdr(fig, "Each part of the reader on its own (absolute values; vertical line = full model)")
    legend_dots(fig, ["full model", "one part removed or changed"], [BLUE, ORANGE], y=-0.04)
    fig.tight_layout(rect=(0, 0.02, 1, 0.95)); fig.subplots_adjust(wspace=0.12)
    save(fig, "05_components", "Ranking-score terms, graph parts and the open segment: absolute Recall@M, best-node IoU and contains-match, with 95% intervals.")
except Exception as e: errs.append(("fig5", repr(e)))
print("errors:", errs)

saved 01_change_vs_full.png  (layout ok)
saved 02_retrieval_vs_answer.png  (layout ok)
saved 03_evidence_budget.png  (layout ok)
saved 04_random_full_oracle.png  (layout ok)
saved 05_components.png  (layout ok)
errors: []


## Step 8: save the results
Zips `/content/out` (summaries, predictions, logs) so it can be downloaded from the Files panel. Do this after each pass: a Colab runtime can be dropped without warning, and everything on its disk is lost.

In [103]:
import textwrap
def check_layout(fig):
    """text boxes that overlap each other (hidden ticks ignored; pairs of rotated tick labels ignored: their boxes overlap but the text does not)"""
    fig.canvas.draw(); r = fig.canvas.get_renderer(); items = []
    for ax in fig.axes:
        x0, x1 = sorted(ax.get_xlim()); y0, y1 = sorted(ax.get_ylim())
        items += ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]
        items += [l for l, loc in zip(ax.get_xticklabels(), ax.get_xticks()) if x0 - 1e-9 <= loc <= x1 + 1e-9]
        items += [l for l, loc in zip(ax.get_yticklabels(), ax.get_yticks()) if y0 - 1e-9 <= loc <= y1 + 1e-9]
        if ax.get_legend(): items += ax.get_legend().get_texts()
    items += fig.texts + [t for lg in fig.legends for t in lg.get_texts()]
    boxes = [(t.get_text(), t.get_window_extent(r), t.get_rotation()) for t in items if t.get_text().strip() and t.get_visible()]
    bad = []
    for i, (a, ba, ra) in enumerate(boxes):
        for b, bb, rb in boxes[i + 1:]:
            if ra % 90 and rb % 90: continue
            ox = min(ba.x1, bb.x1) - max(ba.x0, bb.x0); oy = min(ba.y1, bb.y1) - max(ba.y0, bb.y0)
            if ox > 1 and oy > 1: bad.append(f"overlap: {a[:24]!r} / {b[:24]!r}")
    return bad
def hdr(fig, title, sub=None):
    """title and subtitle placed in inches from the top, so they never collide on short figures; returns the top of the plot area"""
    h = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.08 / h, ha="left", va="top", fontsize=13, fontweight="semibold")
    if sub: fig.text(0.01, 1 - 0.42 / h, sub, ha="left", va="top", fontsize=9.5, color=INK2)
    return 1 - (0.85 if sub else 0.5) / h
QMF = DF[DF.variant == "full"].set_index("query_id")                      # per-question attributes taken from the reference run
BUCKETS = ["<0s", "[0,16)s", "[16,64)s", "[64,256)s", "[256,1024)s"]
def sub_boot(x, qids): return boot(x.reindex(qids))
errs2 = []
# ---- Figure 6: by how far back the question reaches
try:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4), sharex=True); nb = {b: int((QMF.delay_bucket == b).sum()) for b in BUCKETS}; bks = [b for b in BUCKETS if nb[b] >= 10]
    spec = [("recall_at_m", "Retrieval: Recall@M", [("random evidence", "bound_random", GRAY), ("full model", "full", BLUE), ("oracle evidence", "bound_oracle", AQUA)]),
            ("contains", "Answer: contains-match", [("text only", "text_only", ORANGE), ("full model", "full", BLUE), ("oracle evidence", "bound_oracle", AQUA)])]
    for ax, (m, title, ser) in zip(axes, spec):
        for k, (lab, v, c) in enumerate(ser):
            pts = [sub_boot(P[m][v], QMF.index[QMF.delay_bucket == b]) for b in bks]; x = np.arange(len(bks)) + (k - 1) * 0.16
            for xi, (mu, lo, hi) in zip(x, pts): ax.plot([xi, xi], [lo, hi], color=c, lw=2, solid_capstyle="round", zorder=2)
            ax.plot(x, [p[0] for p in pts], "-o", color=c, lw=2, ms=8, mec=SURF, mew=2, label=lab, zorder=3)
        ax.set_xticks(range(len(bks))); ax.set_xticklabels([f"{b}\nn={nb[b]}" for b in bks]); ax.set_ylim(0, 1.0 if m == "recall_at_m" else None); ax.grid(axis="y"); ax.set_axisbelow(True)
        ax.set_title(title, loc="left"); ax.set_xlabel("how long before the question the answer moment happened"); ax.legend(loc="upper right" if m == "contains" else "lower right")
    axes[0].set_ylabel("proportion of questions")
    top = hdr(fig, "Does the memory matter more for older events?", f"Buckets with fewer than 10 questions are left out. {NQ} questions; bars are 95% intervals over streams.")
    fig.tight_layout(rect=(0, 0, 1, top)); save(fig, "06_by_delay", "Recall@M and contains-match by how long before the question the answer moment occurred.")
except Exception as e: errs2.append(("fig6", repr(e)))
# ---- Figure 7: by question template
try:
    cols = [("text only", "text_only"), ("no memory", "M0"), ("blank frame", "blank_frame"), ("full model", "full"), ("8 tokens", "M8"), ("oracle", "bound_oracle")]
    vc = QMF.template.value_counts(); tm = list(vc.index)
    Z = np.array([[P["contains"][v].reindex(QMF.index[QMF.template == t]).mean() for _, v in cols] for t in tm])
    fig, ax = plt.subplots(figsize=(10.5, 0.62 * len(tm) + 2.8)); vmax = max(0.5, np.nanmax(Z)); im = ax.imshow(Z, cmap=SEQ, vmin=0, vmax=vmax, aspect="auto")
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]): ax.text(j, i, f"{Z[i, j]:.2f}", ha="center", va="center", fontsize=9, color="#ffffff" if Z[i, j] > 0.62 * vmax else INK)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([c for c, _ in cols]); ax.xaxis.tick_top()
    ax.set_yticks(range(len(tm))); ax.set_yticklabels([textwrap.fill(f"{t} (n={vc[t]})", 44) for t in tm], fontsize=9)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks(np.arange(-.5, len(cols)), minor=True); ax.set_yticks(np.arange(-.5, len(tm)), minor=True); ax.grid(which="minor", color=SURF, lw=2); ax.tick_params(which="both", length=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02); cb.set_label("contains-match", color=INK2); cb.outline.set_visible(False)
    top = hdr(fig, "Contains-match accuracy by question template", "Rows are templates (question counts in brackets); columns are variants. Small templates are noisy.")
    fig.tight_layout(rect=(0, 0, 1, top)); save(fig, "07_by_template", "Contains-match accuracy for each question template under six variants.")
except Exception as e: errs2.append(("fig7", repr(e)))
# ---- Figure 8: how often two variants give the identical answer
try:
    norm = lambda s: re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()
    NP = PRED.apply(lambda c: c.map(norm)); order = ["full"] + [v for v in ROWS]; k = len(order)
    A = np.array([[(NP[a] == NP[b])[NP[a].notna() & NP[b].notna()].mean() for b in order] for a in order])
    off = A[~np.eye(k, dtype=bool)]; vmin = max(0.0, np.floor(off.min() * 20) / 20)
    fig, ax = plt.subplots(figsize=(11.5, 10.4)); im = ax.imshow(A, cmap=SEQ, vmin=vmin, vmax=1.0)
    for i in range(k):
        for j in range(k): ax.text(j, i, f"{A[i, j] * 100:.0f}", ha="center", va="center", fontsize=7.5, color="#ffffff" if A[i, j] > vmin + 0.62 * (1 - vmin) else INK)
    ax.set_xticks(range(k)); ax.set_xticklabels(order, rotation=45, ha="right", fontsize=9); ax.set_yticks(range(k)); ax.set_yticklabels(order, fontsize=9)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks(np.arange(-.5, k), minor=True); ax.set_yticks(np.arange(-.5, k), minor=True); ax.grid(which="minor", color=SURF, lw=1.5); ax.tick_params(which="both", length=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02); cb.set_label("share of questions with the identical answer", color=INK2); cb.outline.set_visible(False)
    fa = {v: A[0, order.index(v)] for v in order[1:]}
    top = hdr(fig, "How often two variants give the same answer (percent of questions)", f"Against the full model, the median variant gives the identical answer on {np.median(list(fa.values())) * 100:.0f}% of questions (lowest: {min(fa, key=fa.get)}, {min(fa.values()) * 100:.0f}%).")
    fig.tight_layout(rect=(0, 0, 1, top)); save(fig, "08_answer_agreement", "Pairwise share of questions on which two variants produce the same (normalised) answer string.")
except Exception as e: errs2.append(("fig8", repr(e)))
# ---- Figure 9: answer accuracy when retrieval hit vs missed
try:
    vs = ["bound_random", "full", "M8", "bound_oracle"]; fig, ax = plt.subplots(figsize=(8.8, 5.4)); w = 0.12; labels = []
    for i, v in enumerate(vs):
        hit = P["recall_at_m"][v] > 0; ids = {"answer moment retrieved": hit.index[hit], "answer moment not retrieved": hit.index[~hit & P["recall_at_m"][v].notna()]}
        labels.append(f"{NICE[v].replace(' (reference)', '')}\nhit n={len(ids['answer moment retrieved'])}, miss n={len(ids['answer moment not retrieved'])}")
        for k, (lab, q) in enumerate(ids.items()):
            if len(q) < 5: continue
            mu, lo, hi = sub_boot(P["contains"][v], q); x = i + (k - 0.5) * (w + 0.03)
            ax.bar(x, mu, width=w, color=BLUE if k == 0 else GRAY, zorder=2, label=lab if i == 1 else None); ax.plot([x, x], [lo, hi], color=INK2, lw=1, zorder=3)
            ax.annotate(f"{mu:.2f}", (x, hi), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9, color=INK)
    ax.set_xticks(range(len(vs))); ax.set_xticklabels(labels, fontsize=9); ax.set_ylim(0, 0.6); ax.grid(axis="y"); ax.set_axisbelow(True); ax.set_ylabel("contains-match accuracy"); ax.legend(loc="upper left")
    ax.set_title("Is the answer more often right when the answer moment was retrieved?", loc="left", fontsize=12)
    fig.tight_layout(); save(fig, "09_hit_vs_miss", "Contains-match accuracy split by whether the answer moment was among the retrieved evidence (Recall@M above 0), for four variants.")
except Exception as e: errs2.append(("fig9", repr(e)))
# ---- Figure 10: per-stream difference, full minus text only
try:
    d = (P["contains"]["full"] - P["contains"]["text_only"]).groupby(QS).mean().sort_values(); up, dn, eq = (d > 0).sum(), (d < 0).sum(), (d == 0).sum()
    fig, ax = plt.subplots(figsize=(11, 4.8)); cc = [BLUE if x > 0 else RED if x < 0 else GRAY for x in d.values]
    ax.bar(range(len(d)), d.values, width=0.7, color=cc, zorder=2); ax.axhline(0, color=AXIS, lw=1, zorder=1); ax.grid(axis="y"); ax.set_axisbelow(True)
    ax.set_xlabel(f"{len(d)} validation streams, sorted by the difference"); ax.set_ylabel("contains-match: full minus text only"); ax.set_xticks([])
    ax.legend([Line2D([], [], marker="s", ms=8, color=c, lw=0) for c in (BLUE, RED, GRAY)], [f"full better ({up} streams)", f"text only better ({dn} streams)", f"same ({eq} streams)"], loc="upper left")
    ax.set_title("Per stream: does the full model beat answering from the question alone?", loc="left", fontsize=12, pad=22)
    ax.text(0, 1.02, f"Mean difference {d.mean():+.3f}. Streams have only a few questions each, so single bars are noisy.", transform=ax.transAxes, fontsize=9, color=INK2)
    fig.tight_layout(); save(fig, "10_per_stream_difference", "Per-stream contains-match of the full model minus the text-only baseline, sorted.")
except Exception as e: errs2.append(("fig10", repr(e)))
# ---- Figure 11: consistency across the stream groups
try:
    fig, ax = plt.subplots(figsize=(10, 5.2)); con = [("full minus text only", "full", "text_only", BLUE), ("oracle minus full", "bound_oracle", "full", AQUA), ("full minus blank frame", "full", "blank_frame", ORANGE)]
    for k, (lab, a, b, c) in enumerate(con):
        xs_, pts = [], []
        for i, p in enumerate(PIECES):
            q = DF[DF.piece == p].query_id.unique(); mu, lo, hi = boot((P["contains"][a] - P["contains"][b]).reindex(q)); xs_.append(i + (k - 1) * 0.14); pts.append((mu, lo, hi))
        for xi, (mu, lo, hi) in zip(xs_, pts): ax.plot([xi, xi], [lo, hi], color=c, lw=2, solid_capstyle="round", zorder=2)
        ax.plot(xs_, [t[0] for t in pts], "-o", color=c, lw=2, ms=8, mec=SURF, mew=2, label=lab, zorder=3)
    ax.axhline(0, color=AXIS, lw=1, zorder=1); ax.grid(axis="y"); ax.set_axisbelow(True)
    ax.set_xticks(range(len(PIECES))); ax.set_xticklabels([f"streams {int(p[1:4])}-{int(p[5:8]) - 1}" for p in PIECES]); ax.set_ylabel("change in contains-match"); ax.legend(loc="best")
    ax.set_title("Are the main contrasts stable across groups of streams?", loc="left", fontsize=12)
    fig.tight_layout(); save(fig, "11_stability_across_stream_groups", "Three key contrasts in contains-match, computed separately for each group of 10 streams, with 95% intervals.")
except Exception as e: errs2.append(("fig11", repr(e)))
# ---- Figure 12: open versus committed evidence
try:
    locs = [l for l in ["committed", "open", "split"] if (QMF.evidence_location == l).sum() >= 8]; fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0), sharex=True)
    for ax, (m, title) in zip(axes, [("recall_at_m", "Retrieval: Recall@M"), ("contains", "Answer: contains-match")]):
        for k, (lab, v, c) in enumerate([("full model", "full", BLUE), ("open segment hidden", "no_open", ORANGE)]):
            pts = [sub_boot(P[m][v], QMF.index[QMF.evidence_location == l]) for l in locs]; x = np.arange(len(locs)) + (k - 0.5) * 0.16
            for xi, (mu, lo, hi) in zip(x, pts): ax.plot([xi, xi], [lo, hi], color=c, lw=2, solid_capstyle="round", zorder=2)
            ax.plot(x, [p[0] for p in pts], "o", ms=8, color=c, mec=SURF, mew=2, label=lab, zorder=3)
        ax.set_xticks(range(len(locs))); ax.set_xticklabels([f"{l}\nn={(QMF.evidence_location == l).sum()}" for l in locs]); ax.set_ylim(0, 1.0 if m == "recall_at_m" else None)
        ax.grid(axis="y"); ax.set_axisbelow(True); ax.set_title(title, loc="left"); ax.set_xlabel("where the answer moment sits in memory"); ax.legend(loc="best")
    top = hdr(fig, "Where the answer moment sits in memory: finished (committed) vs still being written (open)")
    fig.tight_layout(rect=(0, 0, 1, top)); save(fig, "12_open_vs_committed", "Recall@M and contains-match by where the answer moment lies in memory, with and without the open segment.")
except Exception as e: errs2.append(("fig12", repr(e)))
print("errors:", errs2)

saved 06_by_delay.png  (layout ok)
saved 07_by_template.png  (layout ok)
saved 08_answer_agreement.png  (layout ok)
saved 09_hit_vs_miss.png  (layout ok)
saved 10_per_stream_difference.png  (layout ok)
saved 11_stability_across_stream_groups.png  (layout ok)
saved 12_open_vs_committed.png  (layout ok)
errors: []


In [104]:
METRICS = ["recall_at_m", "t_overlap", "evidence_precision", "best_iou", "exact", "contains"]
for m in METRICS:
    if m not in P: P[m] = piv(m)
for m in list(P): P[m] = P[m].reindex(columns=VAR_ORDER)       # all-empty columns (e.g. evidence precision with no evidence) are dropped by pandas; keep every variant
rows, drows = [], []
for v in VAR_ORDER:
    r = {"variant": v, "description": NICE[v], "n_questions": int(P["contains"][v].notna().sum())}
    for m in METRICS:
        mu, lo, hi = boot(P[m][v]) if P[m][v].notna().sum() else (np.nan,) * 3
        r.update({m: round(mu, 4), m + "_lo": round(lo, 4), m + "_hi": round(hi, 4)})
    rows.append(r)
    if v != "full":
        dr = {"variant": v, "description": NICE[v]}
        for m in ["recall_at_m", "best_iou", "exact", "contains"]:
            mu, lo, hi = delta(m, v); dr.update({"delta_" + m: round(mu, 4), "delta_" + m + "_lo": round(lo, 4), "delta_" + m + "_hi": round(hi, 4)})
        drows.append(dr)
pd.DataFrame(rows).to_csv(f"{PLOTS}/results_table.csv", index=False); pd.DataFrame(drows).to_csv(f"{PLOTS}/deltas_vs_full.csv", index=False)
idx = [f"# Ablation plots\n", f"Generated {time.strftime('%F %T')} UTC from {len(PIECES)} complete stream groups ({', '.join(PIECES)}): {NQ} questions, {NS} validation streams, 18 variants, trained model W_A (evaluation only, no training).\n",
       "Error bars: 95% bootstrap intervals (2000 resamples) that resample whole streams. Differences between variants are paired on the same questions. Colours follow the entity across figures: full model blue, text only orange, oracle evidence aqua, random evidence gray.\n",
       "\n## Figures\n"] + [f"- `{k}.png`: {v}\n" for k, v in sorted(SAVED.items())] + ["\n## Tables\n", "- `results_table.csv`: every metric with 95% intervals, per variant\n", "- `deltas_vs_full.csv`: change vs the full model with 95% intervals\n", "- `report.pdf`: all figures in one file\n"]
open(f"{PLOTS}/INDEX.md", "w").write("".join(idx))
pngs = sorted(glob.glob(f"{PLOTS}/*.png")); pages = []
for f in pngs:
    im = PILImage.open(f).convert("RGB"); w = 1700; pages.append(im.resize((w, int(im.height * w / im.width))))
pages[0].save(f"{PLOTS}/report.pdf", save_all=True, append_images=pages[1:], resolution=150)
sh(f"ls -la {PLOTS} | awk '{{print $5, $9}}'; du -sh {PLOTS}")
print(pd.DataFrame(rows)[["variant", "n_questions", "recall_at_m", "contains", "exact"]].to_string(index=False))

 
4096 .
4096 ..
252999 01_change_vs_full.png
190366 02_retrieval_vs_answer.png
134203 03_evidence_budget.png
67504 04_random_full_oracle.png
146589 05_components.png
184152 06_by_delay.png
276937 07_by_template.png
380019 08_answer_agreement.png
91408 09_hit_vs_miss.png
99965 10_per_stream_difference.png
107064 11_stability_across_stream_groups.png
107059 12_open_vs_committed.png
2143 deltas_vs_full.csv
2231 INDEX.md
1485049 report.pdf
3034 results_table.csv
3.4M	/content/drive/MyDrive/selectstream_ablations/plots

          variant  n_questions  recall_at_m  contains  exact
             full          867       0.5179    0.1995 0.1246
        text_only          867       0.0000    0.1223 0.0092
      blank_frame          867       0.5179    0.1938 0.1211
               M0          867       0.0000    0.1280 0.0000
     bound_random          867       0.4521    0.1984 0.1257
     bound_oracle          867       0.9250    0.1984 0.1257
   score_no_prior          867       0.3968    0.19

## Step 11: finish (run once, after the plots)
Prints the two results tables (so they can be copied out), does a last copy of `/content/out` to Drive, and unmounts Drive so this runtime no longer has access to it.

In [105]:
print(open(f"{PLOTS}/results_table.csv").read()); print("=====DELTAS"); print(open(f"{PLOTS}/deltas_vs_full.csv").read())
SYNC["stop"] = True; print("final sync:", sync())
sh(f"find {DEST}/out -name summary.json | wc -l; ls {PLOTS} | wc -l; du -sh {DEST}")
drive.flush_and_unmount(); print("Drive unmounted:", not os.path.ismount("/content/drive"))

variant,description,n_questions,recall_at_m,recall_at_m_lo,recall_at_m_hi,t_overlap,t_overlap_lo,t_overlap_hi,evidence_precision,evidence_precision_lo,evidence_precision_hi,best_iou,best_iou_lo,best_iou_hi,exact,exact_lo,exact_hi,contains,contains_lo,contains_hi
full,full model (reference),867,0.5179,0.469,0.5621,0.476,0.4248,0.5202,0.0284,0.0212,0.0378,0.1874,0.1478,0.2372,0.1246,0.093,0.1563,0.1995,0.1655,0.2366
text_only,"text only (no frame, no memory)",867,0.0,0.0,0.0,0.0,0.0,0.0,,,,0.2025,0.1693,0.2443,0.0092,0.0039,0.0148,0.1223,0.0963,0.1486
blank_frame,blank frame,867,0.5179,0.4707,0.5613,0.476,0.4271,0.5202,0.0284,0.0215,0.0379,0.1874,0.1485,0.2399,0.1211,0.0937,0.1491,0.1938,0.1613,0.2269
M0,no memory (0 tokens),867,0.0,0.0,0.0,0.0,0.0,0.0,,,,0.2025,0.1687,0.2423,0.0,0.0,0.0,0.128,0.0982,0.1565
bound_random,random evidence,867,0.4521,0.3891,0.5108,0.4063,0.3381,0.4694,0.0214,0.0149,0.0302,0.188,0.1495,0.2379,0.1257,0.0959,0.1569,0.1984,0.1633,0.2341
bound_oracle,oracle evide

In [106]:
print(DF[DF.variant == "full"].participant_disjoint.value_counts().to_dict(), "| linked:", DF[DF.variant == "full"].linked.value_counts().to_dict())

{False: 820, True: 47} | linked: {True: 828, False: 39}


In [99]:
sh("cd /content && rm -f ablation_results.zip && zip -qr ablation_results.zip out && ls -la ablation_results.zip && find out -name summary.json | wc -l")

-rw-r--r-- 1 root root 2204600 Oct 30 03:29 ablation_results.zip
90

